# GRF Training Notebook
**Google Research Football** -- `academy_3_vs_1_with_keeper`

GRF's C++ engine links against Boost.Python. Ubuntu 22.04 (Colab's base)
ships `libboost_python310` but not `libboost_python312`. We therefore:
1. Create a **Python 3.10 venv** so the Boost version matches, and
2. **Build GRF from source** (not PyPI) after patching `build_game_engine.sh`
   to pass `-DPython3_EXECUTABLE` to cmake -- without the patch cmake
   auto-detects the highest Python on the system (3.12) and fails.

## Workflow
1. Run all cells top-to-bottom.
2. Results are saved to Google Drive automatically.
3. `mappo_demo.ipynb` loads them via `DEMO_MODE=True`.

In [ ]:
# ================================================================
# CONFIGURATION -- edit before running
# ================================================================
GITHUB_URL    = 'https://github.com/keckster00/mappo'
DRIVE_RESULTS = '/content/drive/MyDrive/mappo_demo'  # must match main notebook
GRF_VENV      = '/content/grf_venv'                  # Python 3.10 venv for GRF
GRF_SRC       = '/content/football_src'               # gfootball source clone

GRF_STEPS = 5_000_000   # ~30-60 min on A100
N_THREADS = 64           # GRF envs are heavy; 64 is safe on A100

print('Configuration loaded.')
print(f'  GRF_STEPS : {GRF_STEPS:,}')
print(f'  N_THREADS : {N_THREADS}')


In [ ]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')
os.makedirs(DRIVE_RESULTS, exist_ok=True)
print(f'Drive mounted. Results folder: {DRIVE_RESULTS}')

REPO_PATH = '/content/mappo'
if not os.path.exists(REPO_PATH):
    print(f'Cloning repo from {GITHUB_URL}...')
    ret = os.system(f'git clone {GITHUB_URL} {REPO_PATH}')
    if ret != 0:
        raise RuntimeError('git clone failed. Check GITHUB_URL in the config cell.')
else:
    print(f'Repo already at {REPO_PATH}')

os.chdir(REPO_PATH)
print(f'Working directory: {os.getcwd()}')

print('Pulling latest code...')
ret = os.system('git pull origin main')
if ret != 0:
    print('WARNING: git pull failed. Continuing with existing code.')


## Setup
Creates a Python 3.10 venv, clones and patches the GRF source, builds the
C++ engine, then installs PyTorch and the onpolicy package.
**Run once per Colab session** (safe to re-run). Expect ~8-12 minutes.

If the build fails, check the error output from the **'build gfootball engine'** step.

In [ ]:
import subprocess, sys, os, glob, re, tempfile

def run(cmd, label):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    ok = result.returncode == 0
    print(f'  [{"OK" if ok else "FAIL"}] {label}')
    if not ok:
        combined = (result.stdout + result.stderr).strip()
        if combined:
            print(combined[-4000:])
    return ok

def venv_pip(*args):
    return run(f'{GRF_VENV}/bin/python -m pip install -q ' + ' '.join(args),
               ' '.join(args[:2]))

# ── Step 1: System libraries (GRF README list, Ubuntu 22.04 safe) ────────────
print('Step 1: Installing system libraries...')
run('apt-get update -qq > /dev/null 2>&1', 'apt-get update')
run(
    'apt-get install -y '
    'git cmake build-essential '
    'libgl1-mesa-dev libglu1-mesa-dev '
    'libsdl2-dev libsdl2-image-dev libsdl2-ttf-dev libsdl2-gfx-dev '
    'libboost-all-dev '
    'python3.10 python3.10-venv python3.10-dev '
    'xvfb x11vnc '
    '> /dev/null 2>&1',
    'system libraries + python3.10'
)

# ── Step 2: Create Python 3.10 venv ──────────────────────────────────────────
# Ubuntu 22.04 ships libboost_python310; running pip from a 3.10 venv ensures
# Boost links against the right version.
print('Step 2: Creating Python 3.10 venv...')
if not os.path.exists(f'{GRF_VENV}/bin/python'):
    ok = run(f'python3.10 -m venv {GRF_VENV}', 'create venv')
    if not ok:
        raise RuntimeError('python3.10 -m venv failed.')
else:
    print(f'  [OK] venv already exists')
venv_pip('--upgrade pip setuptools psutil wheel')

# ── Step 3: Clone GRF source with full submodules ────────────────────────────
print('Step 3: Cloning gfootball source...')
engine_dir = f'{GRF_SRC}/third_party/gfootball_engine'
if not os.path.exists(engine_dir):
    run(f'rm -rf {GRF_SRC}', 'clean old clone')
    ok = run(
        f'git clone --recurse-submodules '
        f'https://github.com/google-research/football {GRF_SRC}',
        'clone gfootball'
    )
    if not ok:
        raise RuntimeError('git clone failed.')
else:
    print(f'  [OK] already cloned')
print(f'  engine submodule present: {os.path.exists(engine_dir)}')

# ── Step 4: Patch CMakeLists.txt -- CACHE FORCE overrides cmake's Python detection
print('Step 4: Patching CMakeLists.txt...')
inject_lines = [
    'set(Python3_EXECUTABLE "/usr/bin/python3.10" CACHE FILEPATH "" FORCE)',
    'set(Python_EXECUTABLE  "/usr/bin/python3.10" CACHE FILEPATH "" FORCE)',
    'set(PYTHON_EXECUTABLE  "/usr/bin/python3.10" CACHE FILEPATH "" FORCE)',
    'set(Python3_ROOT_DIR   "/usr"               CACHE PATH    "" FORCE)',
]
inject = '\n'.join(inject_lines) + '\n'
for cmake_path in glob.glob(f'{GRF_SRC}/**/CMakeLists.txt', recursive=True):
    with open(cmake_path) as f:
        content = f.read()
    if ('find_package(Python' in content or 'FIND_PACKAGE(Python' in content) and 'python3.10' not in content.lower():
        with open(cmake_path, 'w') as f:
            f.write(inject + content)
        print(f'  patched: {cmake_path}')

# ── Step 4b: Patch build_game_engine.sh -- cmake call gets explicit Python flags
print('Step 4b: Patching build_game_engine.sh...')
build_sh = f'{GRF_SRC}/gfootball/build_game_engine.sh'
if os.path.exists(build_sh):
    with open(build_sh) as f:
        sh = f.read()
    py_flags = (' -DPython3_EXECUTABLE=/usr/bin/python3.10'
                ' -DPYTHON_EXECUTABLE=/usr/bin/python3.10')
    patched = re.sub(r'(cmake\s+)(\$\w+)', r'\1\2' + py_flags, sh)
    if patched != sh:
        with open(build_sh, 'w') as f:
            f.write(patched)
        print(f'  patched: {build_sh}')
    else:
        print(f'  WARNING: cmake line not found -- showing script head:')
        print(sh[:400])
else:
    print(f'  WARNING: build_game_engine.sh not found')

# ── Step 5: Build gfootball from patched source ───────────────────────────────
# --no-build-isolation: pip uses the current environment (venv Python 3.10)
# --no-deps: we install gym separately to pin pre-0.26 version
# No -q: build failures must be visible
print('Step 5: Building gfootball from patched source (~5-8 min)...')
with tempfile.NamedTemporaryFile(mode='w', suffix='.log', delete=False) as lf:
    log_path = lf.name
build_cmd = (
    f'cd {GRF_SRC} && '
    f'{GRF_VENV}/bin/python -m pip install --no-build-isolation --no-deps . '
    f'2>&1 | tee {log_path}'
)
ret = os.system(build_cmd)
if ret != 0:
    print('Build FAILED. Last 80 lines of output:')
    with open(log_path) as f:
        lines = f.readlines()
    print(''.join(lines[-80:]))
    os.unlink(log_path)
    raise RuntimeError('gfootball build failed -- see output above.')
os.unlink(log_path)
print('  [OK] gfootball built successfully')

# ── Step 6: Pin gym and install ML training deps ──────────────────────────────
print('Step 6: Installing ML deps...')
# gym: gfootball may have pulled in a newer version; onpolicy needs 4-tuple step()
venv_pip("'gym==0.25.2'")
venv_pip('torch torchvision --index-url https://download.pytorch.org/whl/cu121')
venv_pip('numpy==1.26.4 scipy imageio tensorboard tensorboardX setproctitle wandb')
venv_pip(f'-e {REPO_PATH}')

print('\nAll setup steps complete.')


In [ ]:
import subprocess, os

GRF_PYTHON = f'{GRF_VENV}/bin/python'

r = subprocess.run([GRF_PYTHON, '--version'], capture_output=True, text=True)
print(f'Venv Python: {r.stdout.strip() or r.stderr.strip()}')

checks = [
    ("import gfootball.env; print('gfootball ok')",                         'gfootball'),
    ("import torch; print(torch.__version__, torch.cuda.is_available())",   'torch + CUDA'),
    ("import gym; print(gym.__version__)",                                   'gym'),
    ("import numpy as np; print(np.__version__)",                            'numpy'),
    ("import onpolicy; print('onpolicy ok')",                                'onpolicy'),
]
all_ok = True
for check, label in checks:
    r = subprocess.run([GRF_PYTHON, '-c', check], capture_output=True, text=True)
    ok = r.returncode == 0
    out = (r.stdout or r.stderr).strip().splitlines()[0] if (r.stdout or r.stderr).strip() else ''
    print(f'  [{"OK" if ok else "FAIL"}] {label}: {out}')
    if not ok:
        all_ok = False
if all_ok:
    print('\nAll checks passed. Safe to run training.')
else:
    raise RuntimeError('Checks failed -- fix errors above before training.')


## Training
A smoke test (1 thread, 2 000 steps) runs first to catch errors before
committing to the full 30-60 min run.

In [ ]:
import subprocess, os, tempfile

os.environ['WANDB_MODE']     = 'disabled'
os.environ['WANDB_DISABLED'] = 'true'

GRF_PYTHON   = f'{GRF_VENV}/bin/python'
RESULTS_BASE = f'{REPO_PATH}/onpolicy/scripts/results'
GRF_SCRIPT   = f'{REPO_PATH}/onpolicy/scripts/train/train_football.py'

# GRF requires a virtual display even in headless training
os.system('pkill Xvfb 2>/dev/null; Xvfb :99 -screen 0 1280x720x24 &')
env = os.environ.copy()
env['DISPLAY']    = ':99'
env['PYTHONPATH'] = REPO_PATH + ':' + env.get('PYTHONPATH', '')

COMMON_ARGS = [
    '--env_name',           'Football',
    '--scenario_name',      'academy_3_vs_1_with_keeper',
    '--num_agents',         '3',
    '--algorithm_name',     'rmappo',
    '--seed',               '1',
    '--n_training_threads', '1',
    '--num_mini_batch',     '1',
    '--episode_length',     '200',
    '--ppo_epoch',          '15',
    '--use_ReLU',
    '--gain',               '0.01',
    '--lr',                 '5e-4',
    '--critic_lr',          '5e-4',
    '--representation',     'simple115v2',
    '--rewards',            'scoring',
    '--use_wandb',          # store_false: disables wandb, uses TensorBoard
]

def run_grf(extra_args, label):
    cmd = [GRF_PYTHON, GRF_SCRIPT] + COMMON_ARGS + extra_args
    with tempfile.NamedTemporaryFile(mode='w', suffix='.log', delete=False) as ef:
        err_path = ef.name
    with open(err_path, 'w') as ef:
        result = subprocess.run(cmd, stderr=ef, text=True, env=env)
    if result.returncode == 0:
        print(f'{label}: PASSED')
        os.unlink(err_path)
    else:
        print(f'{label}: FAILED (exit {result.returncode})')
        with open(err_path) as f:
            lines = f.readlines()
        print('--- last 60 lines of stderr ---')
        print(''.join(lines[-60:]))
        print('--- end ---')
        os.unlink(err_path)
    return result.returncode == 0

# Smoke test: 1 thread x 2 000 steps (~30 s)
print('Running smoke test...')
ok = run_grf([
    '--experiment_name',   'smoke',
    '--n_rollout_threads', '1',
    '--num_env_steps',     '2000',
    '--ppo_epoch',         '1',
    '--save_interval',     '9999',
    '--log_interval',      '1',
], 'Smoke test')
if not ok:
    raise RuntimeError('Smoke test failed -- fix the error above before full training.')
print('Smoke test passed. Starting full training...')
print(f'  steps={GRF_STEPS:,}  threads={N_THREADS}')
print('-' * 60)

ok = run_grf([
    '--experiment_name',   'demo',
    '--n_rollout_threads', str(N_THREADS),
    '--num_env_steps',     str(GRF_STEPS),
    '--save_interval',     '25',
    '--log_interval',      '25',
], 'Full training')

print('-' * 60)
print('Training complete!' if ok else 'Training FAILED -- check stderr above.')


In [ ]:
import shutil, os

RESULTS_BASE = f'{REPO_PATH}/onpolicy/scripts/results'
src = f'{RESULTS_BASE}/Football/academy_3_vs_1_with_keeper/rmappo/demo'
dst = f'{DRIVE_RESULTS}/GRF_3v1'

if os.path.exists(src):
    if os.path.exists(dst):
        shutil.rmtree(dst)
    shutil.copytree(src, dst)
    print(f'Results saved to: {dst}')
    print('Load in mappo_demo.ipynb: set DEMO_MODE=True and run the GRF cell.')
else:
    print(f'No results found at {src}')
    print('Check that training completed successfully.')


## Results -- Training Curve

In [ ]:
import os, glob
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from tensorboard.backend.event_processing.event_accumulator import EventAccumulator

RESULTS_BASE = f'{REPO_PATH}/onpolicy/scripts/results'

def load_scalar(log_dir, tag):
    nested = os.path.join(log_dir, tag, tag)
    target = nested if os.path.isdir(nested) else log_dir
    ea = EventAccumulator(target)
    ea.Reload()
    if tag in ea.Tags().get('scalars', []):
        ev = ea.Scalars(tag)
        return [e.step for e in ev], [e.value for e in ev]
    return [], []

log_dirs = sorted(glob.glob(
    f'{RESULTS_BASE}/Football/academy_3_vs_1_with_keeper/rmappo/demo/run*/logs'))

if not log_dirs:
    print('No logs found. Run the training cell first.')
else:
    steps, values = load_scalar(log_dirs[-1], 'average_episode_rewards')
    if steps:
        fig, ax = plt.subplots(figsize=(9, 5))
        ax.plot(steps, values, color='#4CAF50', linewidth=2)
        ax.set_xlabel('Environment Steps', fontsize=12)
        ax.set_ylabel('Average Episode Reward', fontsize=12)
        ax.set_title('MAPPO -- GRF academy_3_vs_1_with_keeper', fontsize=14, fontweight='bold')
        ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x/1e6:.1f}M'))
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        fig_path = f'{DRIVE_RESULTS}/tc2_grf_curve.png'
        plt.savefig(fig_path, dpi=150, bbox_inches='tight')
        print(f'Final reward: {values[-1]:.4f}')
        print(f'Figure saved: {fig_path}')
        plt.show()
    else:
        print('No scalar data found in', log_dirs[-1])
        ea_root = EventAccumulator(log_dirs[-1]); ea_root.Reload()
        print('Available tags:', ea_root.Tags().get('scalars', []))
        for sub in sorted(os.listdir(log_dirs[-1])):
            print(' ', sub)
